# 2 · Category audit

Accuracy of the automatic label (`category_auto`) against hand labels. Reads the cleaned files only.

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 80)

p = pd.read_csv('../data/amazon_products_deduped.csv')
refs = pd.read_csv('../data/category_reference_labels.csv')
overridden = p['category_auto'] != p['category_regex']
print(f'{len(p)} products; the model changed the keyword label for {overridden.sum()} ({overridden.mean():.1%})')
p['category_source'].value_counts()

7946 products; the model changed the keyword label for 268 (3.4%)


category_source
pipeline            6974
sonnet_confident     505
user                 255
gold                 159
sonnet_unsure         53
Name: count, dtype: int64

## Accuracy

Gold 2 is the clean held-out set (never used to write rules); gold 1 and the random 250 are optimistic (87.6% on the random 250 before the last rule revision).

In [2]:
def score(key):
    e = refs[refs['eval_set'] == key].merge(p[['Title', 'category_regex', 'category_auto']], on='Title')
    return pd.Series({'n': len(e),
                      'keyword rules': f"{(e['category_regex'] == e['label_now']).mean():.1%}",
                      'model (category_auto)': f"{(e['category_auto'] == e['label_now']).mean():.1%}"})

pd.DataFrame({name: score(key) for name, key in
              [('gold 1 (tuned on)', 'gold_1'), ('gold 2 (held out)', 'gold_2'), ('random 250 (user)', 'random_250')]})

,gold 1 (tuned on),gold 2 (held out),random 250 (user)
n,79,80,246
keyword rules,94.9%,92.5%,94.3%
model (category_auto),94.9%,93.8%,95.1%


In [3]:
# remaining misses of the automatic label
ev = refs[refs['eval_set'] != ''].dropna(subset=['eval_set']).merge(p[['Title', 'category_regex', 'category_auto']], on='Title')
ev.loc[ev['category_auto'] != ev['label_now'], ['eval_set', 'Title', 'category_auto', 'label_now']].assign(Title=lambda t: t['Title'].str[:60])

,eval_set,Title,category_auto,label_now
61,random_250,Humminbird LakeMaster Premium Minnesota V1 Map SD Card for F,Marine & GPS,Storage & Memory Cards
93,random_250,"Fender Professional Series Instrument Cable, Guitar Cable 10",Cables & Charging,Audio
103,random_250,Blackmagic Micro Converter BiDirect SDI/HDMI 3G PSU (CONVBDC,Computers & Laptops,Cameras & Photography
108,random_250,"Kasa Smart 3 Way Switch HS210 KIT, Needs Neutral Wire, 2.4GH",Networking,Smart Home & Wearables
138,random_250,"SAMSUNG Galaxy Watch Ultra Trail Band, Smartwatch Replacemen",Phone & Tablet Accessories,Smart Home & Wearables
159,random_250,"Corsair Premium 600W PCIe 5.0 12VHPWR PSU Cable RMX, RMX Shi",Computers & Laptops,Cables & Charging
164,random_250,Elgato Master Mount L - Premium Desk Clamp with Pole extenda,Audio,Cameras & Photography
173,random_250,Microsoft Type Cover for Surface Pro - Black (Renewed),Storage & Memory Cards,Computers & Laptops
192,random_250,HP 230 Wireless Keyboard - Wireless Connection - Low-Profile,Computers & Laptops,Keyboards & Mice
205,random_250,UGREEN Revodok 7 in 1 USB C Hub Gigabit Ethernet Adapter 4K@,Networking,Cables & Charging


## What the model changed

Most common rule → model changes; reading them, most are fixes.

In [4]:
pd.DataFrame({'rule label': p.loc[overridden, 'category_regex'], 'model label': p.loc[overridden, 'category_auto']}).value_counts().head(10)

rule label                  model label               
Other                       Cameras & Photography         27
Computers & Laptops         Audio                         23
Other                       Computers & Laptops           19
                            Smart Home & Wearables        17
                            Audio                         15
Phone & Tablet Accessories  Cables & Charging             14
Computers & Laptops         Cables & Charging             13
Monitors & Displays         Audio                         12
Other                       Storage & Memory Cards         8
                            Phone & Tablet Accessories     6
Name: count, dtype: int64

## Review list: within-category price outliers

Possible misfits; most turn out correct.

In [5]:
lp = np.log(p['current_price'])
grp = lp.groupby(p['category'])
q1, q3 = grp.transform(lambda s: s.quantile(0.25)), grp.transform(lambda s: s.quantile(0.75))
out = (lp < q1 - 1.5 * (q3 - q1)) | (lp > q3 + 1.5 * (q3 - q1))
print(out.sum(), 'price outliers')
p.loc[out, ['Title', 'category', 'category_source', 'current_price']].sort_values(['category', 'current_price']).assign(Title=lambda t: t['Title'].str[:60])

47 price outliers


,Title,category,category_source,current_price
5070,Focal FBATHYSMG Bluetooth® Headphones with Active Noise Canc,Audio,sonnet_confident,1499.00
7496,Motorola Solutions RMU2040 6-Pack Two-Way Radio Digital Non-,Audio,sonnet_confident,1512.00
7365,TUMI - Alpha Bravo Expedition Flap Backpack - Large Carry On,Bags & Cases,sonnet_confident,678.99
7753,"Energizer EN55-12IT, 12 Volt 55 Amp Hr, Long Lasting Sealed",Batteries,sonnet_confident,179.99
7887,"Mighty Max Battery ML210-6GC2-6 Volt 210 AH, Dual Terminal,",Batteries,sonnet_confident,239.99
2419,UGREEN Nexode 500W GaN Fast Charging Station - 6 Ports USB-C,Cables & Charging,sonnet_confident,199.99
7522,Motorola 56531 Multi Unit Charger / Cloning Station,Cables & Charging,sonnet_confident,199.99
3583,Corsair Dual SSD Mounting Bracket (3.5” Internal Drive Bay t,Computers & Laptops,sonnet_confident,4.99
5652,ARCTIC MX-6 (2 g) - Ultimate Performance Thermal Paste for C,Computers & Laptops,sonnet_confident,5.99
3929,4PCS Breadboards Kit Include 2PCS 830 Point 2PCS 400 Point S,Computers & Laptops,sonnet_unsure,6.59
